In [ ]:
"""
Alpha Trading Research Platform

Notebook:
Sprint06_Performance_Analytics

Purpose:
Compare all four strategies on real risk-adjusted numbers instead of
eyeballing a growth chart - CAGR, max drawdown, Sharpe, Sortino,
profit factor, expectancy, win rate, turnover.

Author:
Alison

Version:
0.8
"""

In [ ]:
# =====================================================
# ALPHA v0.8
# Sprint 6 - Performance Analytics
# =====================================================

import matplotlib.pyplot as plt
import pandas as pd

from alpha.config import DEFAULT_CONFIG
from alpha.data import get_prices, get_monthly_prices, get_benchmark_prices
from alpha.portfolio import calculate_monthly_returns
from alpha.regime import calculate_regime
from alpha.backtest import run_backtest
from alpha.analytics import summarize_performance, calculate_max_drawdown

from alpha.strategies.momentum import calculate_monthly_momentum, select_top_stocks, select_bottom_stocks
from alpha.strategies.trend_following import select_trend_positions, select_downtrend_positions
from alpha.strategies.mean_reversion import select_oversold_stocks, select_overbought_stocks
from alpha.strategies.breakout import select_breakout_stocks, select_breakdown_stocks

In [ ]:
config = DEFAULT_CONFIG
config

In [ ]:
# =====================================================
# DATA
# =====================================================

print("Downloading market data...")

prices = get_prices(config)
monthly_prices = get_monthly_prices(prices)
monthly_returns = calculate_monthly_returns(monthly_prices)

print("Downloading benchmark for regime filter...")

benchmark_prices = get_benchmark_prices(config)
monthly_benchmark = benchmark_prices.resample("ME").last()
regime = calculate_regime(monthly_benchmark, config)

In [ ]:
# =====================================================
# RUN EVERY STRATEGY AND SUMMARIZE
# =====================================================

monthly_momentum = calculate_monthly_momentum(monthly_prices, config)

signal_pairs = {
    "Momentum": (
        select_top_stocks(monthly_momentum, config),
        select_bottom_stocks(monthly_momentum, config),
    ),
    "Trend Following": (
        select_trend_positions(monthly_prices, config),
        select_downtrend_positions(monthly_prices, config),
    ),
    "Mean Reversion": (
        select_oversold_stocks(monthly_prices, config),
        select_overbought_stocks(monthly_prices, config),
    ),
    "Breakout": (
        select_breakout_stocks(monthly_prices, config),
        select_breakdown_stocks(monthly_prices, config),
    ),
}

results = {}
summaries = {}

for name, (long_signal, short_signal) in signal_pairs.items():
    result = run_backtest(monthly_returns, long_signal, short_signal, regime, config)
    results[name] = result
    summaries[name] = summarize_performance(result, monthly_prices, config)

In [ ]:
# =====================================================
# COMPARISON TABLE
# =====================================================

summary_table = pd.DataFrame({
    name: {
        "CAGR": s.cagr,
        "Max Drawdown": s.max_drawdown,
        "Sharpe": s.sharpe_ratio,
        "Sortino": s.sortino_ratio,
        "Profit Factor": s.profit_factor,
        "Expectancy": s.expectancy,
        "Win Rate": s.win_rate,
        "Annualized Turnover": s.annualized_turnover,
        "Num Trades": s.num_trades,
    }
    for name, s in summaries.items()
}).T

display(summary_table.style.format({
    "CAGR": "{:.1%}",
    "Max Drawdown": "{:.1%}",
    "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}",
    "Profit Factor": "{:.2f}",
    "Expectancy": "{:.2%}",
    "Win Rate": "{:.1%}",
    "Annualized Turnover": "{:.2f}",
    "Num Trades": "{:.0f}",
}))

In [ ]:
# =====================================================
# DRAWDOWN COMPARISON
# =====================================================
# The number that matters most for whether you can actually stick with
# a strategy - how bad does it get before it recovers.

plt.figure(figsize=(12, 6))

for name, result in results.items():
    _, drawdown = calculate_max_drawdown(result.growth)
    drawdown.plot(label=name)

plt.title("Sprint 6 - Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown from Peak")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =====================================================
# NOTES
# =====================================================
# Profit Factor, Expectancy, and Win Rate come from a reconstructed
# trade log (alpha/analytics.py: build_combined_trade_log), not just
# the monthly returns series - they reflect actual entry/exit pairs
# per stock, for both the long and short leg.
#
# Sharpe and Sortino use config.risk_free_rate (annual). Worth setting
# this to something realistic (e.g. current T-bill yield) rather than
# leaving it at the 0.0 default if you want the ratios to mean
# something precise.
#
# Next is Sprint 7 - Scanning Engine, which ranks opportunities across
# strategies automatically instead of running each notebook by hand.